In [2]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from PIL import Image
import requests

import json
import re
from pathlib import Path

import pandas as pd
import torch

## Load Microsoft Model

In [3]:
from transformers.models.trocr.modeling_trocr import TrOCRSinusoidalPositionalEmbedding

MODEL_NAME = "microsoft/trocr-large-printed"

processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(
    MODEL_NAME,
    low_cpu_mem_usage=False,
    device_map=None,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Patch positional embedding to avoid meta tensor outputs in some environments.
old_ep = model.decoder.model.decoder.embed_positions
num_positions = int(old_ep.weights.shape[0] - old_ep.offset)
new_ep = TrOCRSinusoidalPositionalEmbedding(
    num_positions=num_positions,
    embedding_dim=old_ep.embedding_dim,
    padding_idx=old_ep.padding_idx,
).to(device)
model.decoder.model.decoder.embed_positions = new_ep

print(f"Using device: {device}")

Loading weights: 100%|██████████| 632/632 [00:00<00:00, 3523.12it/s]
VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-large-printed
Key                                                 | Status     | 
----------------------------------------------------+------------+-
decoder.model.decoder.embed_positions._float_tensor | UNEXPECTED | 
encoder.pooler.dense.weight                         | MISSING    | 
encoder.pooler.dense.bias                           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Using device: cpu


## Build Test Crop Index

Load test JSON annotations, resolve crop image paths, and create a per-box dataframe.

In [4]:
# build crop index from the DEV split json
WORKSPACE = Path.cwd()
JSON_PATH = WORKSPACE / "mlt_data" / "ocr_data_v11_dev.json"
CROPS_DIR = WORKSPACE / "mlt_data" / "crops" / "dev"

print(f"JSON exists: {JSON_PATH.exists()} -> {JSON_PATH}")
print(f"Crops dir exists: {CROPS_DIR.exists()} -> {CROPS_DIR}")

with JSON_PATH.open("r", encoding="utf-8") as f:
    raw = json.load(f)

if isinstance(raw, dict):
    records = raw.get("data", [])
elif isinstance(raw, list):
    records = raw
else:
    raise TypeError("Unsupported json structure for OCR dev data")

rows = []
for rec in records:
    page_obj = rec.get("page", {}) if isinstance(rec, dict) else {}
    page_fname = str(
        rec.get("page_fname", page_obj.get("page_fname", rec.get("image", rec.get("filename", ""))))
    )
    if not page_fname:
        continue

    page_stem = Path(page_fname).stem
    if "-" in page_stem:
        default_doc, default_page = page_stem.split("-", 1)
    else:
        default_doc, default_page = page_stem, ""

    doc_id = str(rec.get("document_id", page_obj.get("document_id", default_doc)))
    page_id = str(rec.get("page_id", page_obj.get("page_id", default_page)))

    boxes = rec.get("boxes", rec.get("bb", rec.get("bboxes", []))) if isinstance(rec, dict) else []
    for box_idx, box in enumerate(boxes):
        if isinstance(box, dict):
            gt_text = str(
                box.get("text", box.get("gt_text", box.get("transcription", box.get("value", ""))))
            )
        else:
            gt_text = str(box)

        crop_path = CROPS_DIR / doc_id / page_id / f"{page_stem}_box{box_idx:04d}.jpg"
        rows.append(
            {
                "page_fname": page_fname,
                "doc_id": doc_id,
                "page_id": page_id,
                "box_idx": box_idx,
                "gt_text": gt_text,
                "crop_path": str(crop_path),
                "exists": crop_path.exists(),
            }
        )

expected_cols = ["page_fname", "doc_id", "page_id", "box_idx", "gt_text", "crop_path", "exists"]
df_crops = pd.DataFrame(rows)
if df_crops.empty:
    df_crops = pd.DataFrame(columns=expected_cols)
else:
    df_crops = df_crops.sort_values(["page_fname", "box_idx"]).reset_index(drop=True)

print(f"Crop records parsed: {len(df_crops)}")
print(f"Missing crop images: {(~df_crops['exists']).sum() if 'exists' in df_crops.columns else 0}")
df_crops.head(12)

JSON exists: True -> /home/user/Desktop/Nomocrat/Evaluation/Microsoft/mlt_data/ocr_data_v11_dev.json
Crops dir exists: True -> /home/user/Desktop/Nomocrat/Evaluation/Microsoft/mlt_data/crops/dev
Crop records parsed: 587
Missing crop images: 0


,page_fname,doc_id,page_id,box_idx,gt_text,crop_path,exists
0,0003-001.jpg,0003,001,0,IT-TISĦIĦ TA’ REŻILJENZA PERMEZZ TA’ SISTEMI T...,/home/user/Desktop/Nomocrat/Evaluation/Microso...,True
1,0003-001.jpg,0003,001,1,Gwida għall-Istabbiliment ta’ Kultura ta’ Komu...,/home/user/Desktop/Nomocrat/Evaluation/Microso...,True
2,0003-001.jpg,0003,001,2,L-Aġenzija Ewropea għall-Ħtiġijiet Speċjali u ...,/home/user/Desktop/Nomocrat/Evaluation/Microso...,True
3,0005-003.jpg,0005,003,0,2,/home/user/Desktop/Nomocrat/Evaluation/Microso...,True
4,0005-003.jpg,0005,003,1,Tradott minn: Tarcisio Zarb,/home/user/Desktop/Nomocrat/Evaluation/Microso...,True
5,0005-003.jpg,0005,003,2,L-istampa tal-faċċata: collage mix-xoghol ta’ ...,/home/user/Desktop/Nomocrat/Evaluation/Microso...,True
6,0005-003.jpg,0005,003,3,L-Aġenzija Ewropea għall-Iżvilupp ta’ Edukazzj...,/home/user/Desktop/Nomocrat/Evaluation/Microso...,True
7,0005-003.jpg,0005,003,4,ll-fehmiet espressi minn individwi f’dan id-do...,/home/user/Desktop/Nomocrat/Evaluation/Microso...,True
8,0005-003.jpg,0005,003,5,Verżjonijiet elettroniċi ta’ dan ir-rapport b’...,/home/user/Desktop/Nomocrat/Evaluation/Microso...,True
9,0005-003.jpg,0005,003,6,ISBN: 978-87-92387-68-4 (Elettronika),/home/user/Desktop/Nomocrat/Evaluation/Microso...,True


## Run Microsoft OCR

In [5]:
# run OCR on every available crop image
to_score = df_crops[df_crops["exists"]].copy() if "exists" in df_crops.columns else df_crops.copy()

print(f"Rows in df_crops: {len(df_crops)}")
print(f"Rows with existing crop files: {len(to_score)}")

# recover automatically if model ended up on meta tensors
model_device = next(model.parameters()).device
if model_device.type == "meta":
    print("Model is on meta device. Reloading model on a real device...")
    model = VisionEncoderDecoderModel.from_pretrained(
        MODEL_NAME,
        low_cpu_mem_usage=False,
        device_map=None,
    )
    model.to(device)

pred_rows = []
failed_paths = []
model.eval()

with torch.no_grad():
    for row in to_score.itertuples(index=False):
        crop_path = Path(row.crop_path)
        try:
            image = Image.open(crop_path).convert("RGB")
            pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device)
            generated_ids = model.generate(pixel_values, max_new_tokens=128)
            pred_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        except Exception as err:
            pred_text = ""
            failed_paths.append((str(crop_path), str(err)))

        pred_rows.append(
            {
                "page_fname": row.page_fname,
                "doc_id": row.doc_id,
                "page_id": row.page_id,
                "box_idx": row.box_idx,
                "pred_text": pred_text,
            }
        )

df_preds = pd.DataFrame(pred_rows)
df_eval = df_crops.merge(
    df_preds,
    on=["page_fname", "doc_id", "page_id", "box_idx"],
    how="left",
)
df_eval["pred_text"] = df_eval["pred_text"].fillna("")

# exact match after light whitespace normalization
norm = lambda s: re.sub(r"\s+", " ", str(s)).strip()
df_eval["gt_norm"] = df_eval["gt_text"].map(norm)
df_eval["pred_norm"] = df_eval["pred_text"].map(norm)
df_eval["exact_match"] = df_eval["gt_norm"] == df_eval["pred_norm"]

print(f"Predictions generated: {(df_eval['pred_text'] != '').sum()}")
print(f"Exact match rate: {df_eval['exact_match'].mean():.4f}")
print(f"Failed OCR rows: {len(failed_paths)}")
if failed_paths:
    print("First 5 failures:")
    for p, e in failed_paths[:5]:
        print(f"- {p}: {e}")

df_eval[["page_fname", "box_idx", "gt_text", "pred_text", "exact_match"]].head(20)

Rows in df_crops: 587
Rows with existing crop files: 587
Predictions generated: 587
Exact match rate: 0.1039
Failed OCR rows: 0


,page_fname,box_idx,gt_text,pred_text,exact_match
0,0003-001.jpg,0,IT-TISĦIĦ TA’ REŻILJENZA PERMEZZ TA’ SISTEMI T...,PERMASTA'S ESTIMERA,False
1,0003-001.jpg,1,Gwida għall-Istabbiliment ta’ Kultura ta’ Komu...,GONDA SHALL-ISTEBIMETIVE' KUTURA TAY,False
2,0003-001.jpg,2,L-Aġenzija Ewropea għall-Ħtiġijiet Speċjali u ...,L-AGENZIJA EUROPEA GRALL-HTIGIJET SPECJALI U I...,False
3,0005-003.jpg,0,2,2,True
4,0005-003.jpg,1,Tradott minn: Tarcisio Zarb,TRADOTT MINN: TARCISIO ZARB,False
5,0005-003.jpg,2,L-istampa tal-faċċata: collage mix-xoghol ta’ ...,LUSTAMARIRE SALES ARE BETTER COMEEMANABLE FROM...,False
6,0005-003.jpg,3,L-Aġenzija Ewropea għall-Iżvilupp ta’ Edukazzj...,"INCLUDENT AND LETCOME, INCLUSIVE INTERNATIONAL...",False
7,0005-003.jpg,4,ll-fehmiet espressi minn individwi f’dan id-do...,"INWARRIED SMEAD, MARINA, TAMPOI, MARINA, JOHOR...",False
8,0005-003.jpg,5,Verżjonijiet elettroniċi ta’ dan ir-rapport b’...,YERZICY.WYWWEETTOPICIAN AGENCY/GOTT B/21 INGWA...,False
9,0005-003.jpg,6,ISBN: 978-87-92387-68-4 (Elettronika),ISBN: 978-87-92387-68-4 (ELETTRONIKA),False


## Calculate CER

In [7]:
# CER evaluation at crop level (same style as tesseract_mlt_test.ipynb)
import unicodedata

def normalize_text(s: str) -> str:
    s = unicodedata.normalize("NFC", str(s))
    s = " ".join(s.split())
    s = s.casefold() # case-insensitive
    return s

def levenshtein_distance(a: str, b: str) -> int:
    if a == b:
        return 0
    if len(a) == 0:
        return len(b)
    if len(b) == 0:
        return len(a)

    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        curr = [i]
        for j, cb in enumerate(b, start=1):
            ins = curr[j - 1] + 1
            dele = prev[j] + 1
            sub = prev[j - 1] + (ca != cb)
            curr.append(min(ins, dele, sub))
        prev = curr
    return prev[-1]

# use existing df_eval from OCR cell and add CER columns
df_eval = df_eval.copy()
df_eval["gt_text"] = df_eval["gt_text"].fillna("")
df_eval["pred_text"] = df_eval["pred_text"].fillna("")
df_eval["gt_norm"] = df_eval["gt_text"].map(normalize_text)
df_eval["pred_norm"] = df_eval["pred_text"].map(normalize_text)

df_eval["gt_len"] = df_eval["gt_norm"].str.len()
df_eval["edit_distance"] = [
    levenshtein_distance(pred, gt)
    for pred, gt in zip(df_eval["pred_norm"], df_eval["gt_norm"])
]
df_eval["cer"] = df_eval["edit_distance"] / df_eval["gt_len"].replace(0, pd.NA)

weighted_cer = df_eval["edit_distance"].sum() / max(df_eval["gt_len"].sum(), 1)
mean_cer = df_eval["cer"].dropna().mean()

print(f"Evaluated crops: {len(df_eval)}")
print(f"Weighted CER: {weighted_cer:.4f}")
print(f"Mean CER (per-crop): {mean_cer:.4f}")

df_eval[["page_fname", "box_idx", "gt_len", "edit_distance", "cer", "gt_text", "pred_text"]].sort_values("cer", ascending=False).reset_index(drop=True)

Evaluated crops: 587
Weighted CER: 0.8139
Mean CER (per-crop): 0.3116


,page_fname,box_idx,gt_len,edit_distance,cer,gt_text,pred_text
0,0078-005.jpg,5,137,137,1.0,Inti titkellem dwar il-fatt li hija l-popolari...,***
1,0078-005.jpg,6,770,770,1.0,Ġieli jistaqsuni kif ma teżistix akkademja tal...,***
2,0100-002.jpg,16,451,451,1.0,"Dan ma nagħmlu la b'sens ta’ akkuża, u lanqas ...",***
3,0060-043.jpg,0,138,138,1.0,"Mikelanġ Camilleri, il-qassis ribell u t-tradu...",***
4,0060-043.jpg,2,711,711,1.0,"Hekk kif ġie lura l-Birgu, il-parroċċa tal-ħid...",***
...,...,...,...,...,...,...,...
582,0069-007.jpg,22,22,0,0.0,element normattiv 9(2),ELEMENT NORMATTIV 9(2)
583,0069-007.jpg,21,19,0,0.0,ECLAS 5(11); 14(19),ECLAS 5(11); 14(19)
584,0069-007.jpg,19,21,0,0.0,"Dritt Ruman, id- 5(7)","DRITT RUMAN, ID- 5(7)"
585,0069-007.jpg,35,33,0,0.0,"Essays on de Soldanis, 2010 9(25)","ESSAYS ON DE SOLDANIS, 2010 9(25)"


## Analyzing Page Outputs

In [14]:
# Visualize crops with GT and prediction for a specific page, plus page-level weighted CER
from IPython.display import display, Markdown

PAGE_ID = "0100-002"  # e.g. 0003-003 or 0003-003.jpg
MAX_SHOW = None  # Set to None to show all crops for that page

target_page = PAGE_ID if PAGE_ID.endswith(".jpg") else f"{PAGE_ID}.jpg"
page_rows = df_eval[df_eval["page_fname"] == target_page].sort_values("box_idx").reset_index(drop=True)

if MAX_SHOW is not None:
    page_rows = page_rows.head(MAX_SHOW)

print(f"Page: {target_page}")
print(f"Crops shown: {len(page_rows)}")

if page_rows.empty:
    print("No rows found. Check PAGE_ID and ensure OCR/CER cells were run first.")
else:
    total_edits = page_rows["edit_distance"].sum()
    total_gt_len = int(page_rows["gt_len"].sum())
    page_weighted_cer = total_edits / max(total_gt_len, 1)
    page_mean_cer = page_rows["cer"].dropna().mean()

    print(f"Weighted CER for page: {page_weighted_cer:.4f}")
    print(f"Mean CER for page (per-crop): {page_mean_cer:.4f}")
    print("=" * 80)

    # for _, row in page_rows.iterrows():
    #     crop_path = Path(row["crop_path"])

    #     display(Markdown(f"### box {int(row['box_idx']):04d} | CER: {float(row['cer']):.4f}"))
    #     display(Image.open(crop_path))
    #     display(Markdown("**GT**"))
    #     print(str(row["gt_text"]))
    #     display(Markdown("**Prediction**"))
    #     print(str(row["pred_text"]))
    #     print("-" * 80)

Page: 0100-002.jpg
Crops shown: 19
Weighted CER for page: 0.9407
Mean CER for page (per-crop): 0.5864


In [15]:
# Excel-friendly TSV output: one row per BB with GT, Prediction, and CER columns

def join_lines(lines: list[str]) -> str:
    if lines == []:
        return ""

    lines = [
        line
        for line in (
            line.strip()
            for line in lines
        )
        if line != ""
    ]

    if lines == []:
        return ""

    new_lines = []
    last_line_index = len(lines) - 1
    for i, line in enumerate(lines):
        new_lines.append(line)
        if (
            i != last_line_index and (
                line[-1] not in "-—"
                or (len(line) > 1 and line[-2] == " ")
            )
        ):
            new_lines.append(" ")

    return "".join(new_lines)


def _clean_text(v):
    if v is None:
        return ""
    return join_lines(str(v).splitlines())

# Prefer the currently selected page; fall back to all evaluated rows.
if "page_rows" in globals() and page_rows is not None and len(page_rows) > 0:
    src = page_rows.copy()
else:
    src = df_eval.copy()

needed = [c for c in ["page_fname", "box_idx", "gt_text", "pred_text", "cer"] if c in src.columns]
out = src[needed].copy()

if "page_fname" not in out.columns:
    out["page_fname"] = ""
if "box_idx" not in out.columns:
    out["box_idx"] = range(len(out))
if "cer" not in out.columns:
    out["cer"] = pd.NA

out = out.sort_values(["page_fname", "box_idx"], kind="stable")

export_df = out.assign(
    value=out.apply(lambda r: f"page:{_clean_text(r.get('page_fname', ''))} bb:{int(r.get('box_idx', 0)):04d}", axis=1),
    GT=out["gt_text"].map(_clean_text),
    Prediction=out["pred_text"].map(_clean_text),
    CER=out["cer"].map(lambda x: "" if pd.isna(x) else f"{float(x):.4f}"),
)[["value", "GT", "Prediction", "CER"]]

# Print as tab-separated values so copy/paste lands in Excel columns.
print(export_df.to_csv(sep="\t", index=False))

value	GT	Prediction	CER
page:0100-002.jpg bb:0000	2	2	0.0000
page:0100-002.jpg bb:0001	l-aċċent	I-ACCENT	0.3750
page:0100-002.jpg bb:0002	ListCountriesandtheirDerivatives.xls	LISTCOUNTRIESANDTHEIRDERIVATIVES.XLS	0.0000
page:0100-002.jpg bb:0003	Xi ħaġa uffiċjali waslitilna, mhux mill-Kunsill tal-Malti, imma direttament mill-Gvern Malti. Fin-nuqqas ta' xi ħaġa aktar uffiċjali minnha (jekk tabilħaqq jista' jkun hemm xi ħaġa aktar uffiċjali minn struzzjonijiet diretti mill-Gvern), ikollna nimxu magħha f'xogħolna.	***	1.0000
page:0100-002.jpg bb:0004	"Niġu għal-lista, li tqassmitilna bħala Excel file bl-isem ""ListCountriesandtheirDerivatives.xls""."	"BL-ISHAN ""LISTA, LIT, JALAN INHOR DETAILA EXCEI XLS""."	0.7010
page:0100-002.jpg bb:0005	Meta ftaħtha l-ewwel darba kont sorpriż. Mhux tant għall-fatt li l-isem tal-fajl ta' lista ta' ismijiet bil-Malti kellu jkun bl-Ingliż, u lanqas għall-fatt li l-pajjiżi jidhru fl-ordni tal-alfabet Ingliż¹. Kont sorpriż bil-lista għaliex mal-ewwel daqqa t

## Worst-Page CER Analysis
Automatically find the page with the highest weighted CER and analyze it using the same page-level view as Section 7.

In [16]:
from IPython.display import display, Markdown

page_summary = (
    df_eval.groupby("page_fname", dropna=False)
    .agg(
        total_edits=("edit_distance", "sum"),
        total_gt_len=("gt_len", "sum"),
        mean_cer=("cer", "mean"),
        num_boxes=("box_idx", "count"),
    )
    .reset_index()
)
page_summary["weighted_cer"] = page_summary["total_edits"] / page_summary["total_gt_len"].clip(lower=1)
page_summary = page_summary.sort_values(["weighted_cer", "mean_cer", "page_fname"], ascending=[False, False, True]).reset_index(drop=True)

worst_page = str(page_summary.loc[0, "page_fname"])
worst_page_rows = df_eval[df_eval["page_fname"] == worst_page].sort_values("box_idx").reset_index(drop=True)
worst_page_weighted_cer = float(page_summary.loc[0, "weighted_cer"])
worst_page_mean_cer = float(page_summary.loc[0, "mean_cer"])
worst_page_boxes = int(page_summary.loc[0, "num_boxes"])

print(f"Worst page by weighted CER: {worst_page}")
print(f"Crops shown: {worst_page_boxes}")
print(f"Weighted CER for page: {worst_page_weighted_cer:.4f}")
print(f"Mean CER for page (per-crop): {worst_page_mean_cer:.4f}")
print("=" * 80)

for _, row in worst_page_rows.iterrows():
    crop_path = Path(row["crop_path"])

    # display(Markdown(f"### box {int(row['box_idx']):04d} | CER: {float(row['cer']):.4f}"))
    # display(Image.open(crop_path))
    # display(Markdown("**GT**"))
    # print(str(row["gt_text"]))
    # display(Markdown("**Prediction**"))
    # print(str(row["pred_text"]))
    # print("-" * 80)

Worst page by weighted CER: 0005-024.jpg
Crops shown: 6
Weighted CER for page: 0.9931
Mean CER for page (per-crop): 0.6667


In [17]:
PAGE_ID = "0005-024"  # e.g. 0003-003 or 0003-003.jpg
MAX_SHOW = None  # Set to None to show all crops for that page

target_page = PAGE_ID if PAGE_ID.endswith(".jpg") else f"{PAGE_ID}.jpg"
page_rows = df_eval[df_eval["page_fname"] == target_page].sort_values("box_idx").reset_index(drop=True)


In [18]:
# Excel-friendly TSV output: one row per BB with GT, Prediction, and CER columns

def join_lines(lines: list[str]) -> str:
    if lines == []:
        return ""

    lines = [
        line
        for line in (
            line.strip()
            for line in lines
        )
        if line != ""
    ]

    if lines == []:
        return ""

    new_lines = []
    last_line_index = len(lines) - 1
    for i, line in enumerate(lines):
        new_lines.append(line)
        if (
            i != last_line_index and (
                line[-1] not in "-—"
                or (len(line) > 1 and line[-2] == " ")
            )
        ):
            new_lines.append(" ")

    return "".join(new_lines)


def _clean_text(v):
    if v is None:
        return ""
    return join_lines(str(v).splitlines())

# Prefer the currently selected page; fall back to all evaluated rows.
if "page_rows" in globals() and page_rows is not None and len(page_rows) > 0:
    src = page_rows.copy()
else:
    src = df_eval.copy()

needed = [c for c in ["page_fname", "box_idx", "gt_text", "pred_text", "cer"] if c in src.columns]
out = src[needed].copy()

if "page_fname" not in out.columns:
    out["page_fname"] = ""
if "box_idx" not in out.columns:
    out["box_idx"] = range(len(out))
if "cer" not in out.columns:
    out["cer"] = pd.NA

out = out.sort_values(["page_fname", "box_idx"], kind="stable")

export_df = out.assign(
    value=out.apply(lambda r: f"page:{_clean_text(r.get('page_fname', ''))} bb:{int(r.get('box_idx', 0)):04d}", axis=1),
    GT=out["gt_text"].map(_clean_text),
    Prediction=out["pred_text"].map(_clean_text),
    CER=out["cer"].map(lambda x: "" if pd.isna(x) else f"{float(x):.4f}"),
)[["value", "GT", "Prediction", "CER"]]

# Print as tab-separated values so copy/paste lands in Excel columns.
print(export_df.to_csv(sep="\t", index=False))

value	GT	Prediction	CER
page:0005-024.jpg bb:0000	Ċertament, xi wħud jinsistu, li f’xi każi, assessjar inizjali jista’ jsir ukoll mingħajr l-użu tal-lingwa. Madanakollu, hemm differenzi kulturali (per eżempju fil-livell tad-definizzjoni tal-kuluri jew tal-kuntest tat-tagħlim, kif intqal minn Salameh, 2006) li jista’ jkollu effett fuq l-proċessi tal-assessjar mhux-verbali jekk dawn ma jittiħdux inkonsiderazzjoni.	***	1.0000
page:0005-024.jpg bb:0001	Fid-dawl tal-parteċipazzjoni tal-istudenti bi sfond ta’ immigrant fl-edukazzjoni speċjali, b’mod sproporzjonat daqshekk sinifikanti bħala l-punt tat-tluq, l-istudji ddubitaw dwar il-kwalità (l-aktar inizjali) tal-proċeduri tal-assessjar li jsiru ma’ dawn l-istudenti bi sfond ta’ immigrant (Andersson, 2007; Rosenqvist, 2007). Metodi u għodod ta’ assessjar bħal dawn ħafna drabi jkunu mniżżla fil-kultura tal-pajjiż tar-residenza u l-iskola tal-istudent/a. Il-proċess ta’ assessjar għalhekk huwa kulturalment preġudikat; it-tfal b’kultura differen